# 原 Stage1 39 项 + TSMC 原始前瞻营收 H5 信息
冻结同样账户父链和 SOXX 终门；本单元先做不拟合的来源、成熟度、训练新颖性与 DatasetH 预检。

In [1]:
import hashlib,json,os,sys
from pathlib import Path
import numpy as np,pandas as pd,qlib,sklearn
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd().resolve();card=root/'research/stage1_tsmc_guidance_growth_h5_information_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and sys.version_info[:2]==(3,12) and sklearn.__version__=='1.9.1'
assert not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
parent=plan['parent'];pc=root/parent['source_card']
for artifact,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:assert sha(pc/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/artifact)==parent[key]
assert json.loads((root/parent['nas_readback']).read_text())['NAS_archive_sha256']==parent['nas_archive_sha256']
src=plan['source'];sc=root/src['card'];ss=json.loads((sc/'summary.json').read_text());assert sha(sc/'summary.json')==src['summary_sha256'] and ss['recorder_id']==src['recorder_id'] and ss['experiment_id']==src['experiment_id'] and ss['nas_evidence']['sha256']==src['nas_archive_sha256'] and ss['nas_readback']['verified']
assert sha(sc/'qlib_recorder'/src['experiment_id']/src['recorder_id']/'artifacts'/src['feature_artifact'])==src['feature_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(pc/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns) and frame.index.equals(base.index) and frame.index.equals(timing.index)
with R.uri_context(ss['recorder_uri']):mapped=R.get_recorder(recorder_id=src['recorder_id'],experiment_id=src['experiment_id']).load_object(src['feature_artifact'])
assert len(mapped)==len(frame)==3966 and mapped.index.equals(frame.index) and int(mapped.loc['2011-01-03':'2022-12-30','source_known'].sum())==src['train_calendar_known'] and int(mapped.loc['2023-01-03':'2026-10-07','source_known'].sum())==src['held_calendar_known']
feature=mapped[src['feature']].astype(float).rename(src['feature']);assert feature.loc[mapped.source_known].notna().all()
forward=frame.soxx_forward_open_return.astype(float);parts=[forward.shift(-i) for i in range(5)];valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts]);label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True);cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
cohort=plan['cohorts'];train_dates=(frame.index>=pd.Timestamp(cohort['train_signal'][0]))&(frame.index<=pd.Timestamp(cohort['train_signal'][1]))&(mature<cutoff)&label.notna();held_dates=(frame.index>=pd.Timestamp(cohort['held_reused_diagnostic'][0]))&(frame.index<=pd.Timestamp(cohort['held_reused_diagnostic'][1]))&(mature<last)&label.notna()
assert int(train_dates.sum())==cohort['expected_mature_train_rows'] and int(held_dates.sum())==cohort['expected_mature_held_rows']
assert feature.loc[held_dates].notna().all() and np.isfinite(feature.loc[held_dates]).all()
known_train=train_dates&mapped.source_known;maxcorr=float(base.loc[known_train,base_names].corrwith(feature.loc[known_train]).abs().max());assert maxcorr<plan['single_mechanism']['training_only_max_abs_pairwise_Pearson_stop_at_or_above']
aug=base[base_names].copy();aug[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates,'source_known':mapped.source_known},index=frame.index)
static=pd.concat({'feature':aug,'label':label.to_frame(),'meta':meta},axis=1);static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(cohort['train_signal']),'held':tuple(cohort['held_reused_diagnostic'])});tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(tr)==cohort['expected_train_calendar_rows'] and len(he)==cohort['expected_held_calendar_rows']
report={'status':'conditional_TSMC_guidance_H5_DatasetH_source_maturity_novelty_preflight_passed_no_fit','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['recorder_id'],'parent_recorder_id':parent['recorder_id'],'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'train_segment_rows':len(tr),'held_segment_rows':len(he),'source_known_train_calendar':src['train_calendar_known'],'source_known_held_calendar':src['held_calendar_known'],'candidate_train_unknown_mature_rows':int(feature.loc[train_dates].isna().sum()),'candidate_held_unknown_mature_rows':int(feature.loc[held_dates].isna().sum()),'max_abs_training_pairwise_Pearson':maxcorr,'parent_Recorder_object_loads':4,'source_Recorder_object_loads':1,'DatasetH_builds':1,'model_fits':0,'native_accounts':0,'historical_first_public_exact_clock_qualified':False}
receipt=card/'preflight_result.json'
if receipt.exists():assert json.loads(receipt.read_text())==report
else:receipt.write_text(json.dumps(report,indent=2)+'\n')
print({k:report[k] for k in ('status','training_mature_rows','held_mature_rows','candidate_train_unknown_mature_rows','max_abs_training_pairwise_Pearson')})

{'status': 'conditional_TSMC_guidance_H5_DatasetH_source_maturity_novelty_preflight_passed_no_fit', 'training_mature_rows': 3014, 'held_mature_rows': 940, 'candidate_train_unknown_mature_rows': 18, 'max_abs_training_pairwise_Pearson': 0.11683956342141448}


In [2]:
import shlex,subprocess,tarfile,tempfile
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings
report=json.loads((card/'preflight_result.json').read_text());assert report['plan_sha256']==sha(card/'plan.json') and report['model_fits']==report['native_accounts']==0
assert report['training_mature_rows']==int(train_dates.sum()) and report['held_mature_rows']==int(held_dates.sum()) and report['max_abs_training_pairwise_Pearson']==maxcorr
assert not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
train=tr.loc[tr[('meta','train_eligible')].astype(bool)];held=he.loc[he[('meta','held_eligible')].astype(bool)]
assert len(train)==report['training_mature_rows']==3014 and len(held)==report['held_mature_rows']==940
ytr=train[('label','soxx_forward_five_open_log_return')].to_numpy(float);yhe=held[('label','soxx_forward_five_open_log_return')].to_numpy(float);assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
name=src['feature'];inputs={'baseline':(train['feature'][base_names].to_numpy(float),held['feature'][base_names].to_numpy(float)),'candidate':(train['feature'][base_names+[name]].to_numpy(float),held['feature'][base_names+[name]].to_numpy(float))}
assert np.isfinite(inputs['candidate'][1][:,-1]).all() and not np.isinf(inputs['candidate'][0][:,-1]).any()
state={'status':'fit_attempting','plan_sha256':sha(card/'plan.json'),'fit_attempts':[],'fits_completed':[],'bootstrap_attempts':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
models={};predictions={};tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,dataset=dataset,derived_coordinate=feature,source_hashes={'original39':parent['feature_artifact_sha256'],'TSMC_guidance':src['feature_artifact_sha256']})
 for family in ('baseline','candidate'):
  assert len(state['fit_attempts'])<plan['model']['fit_attempts_max'] and family not in state['fit_attempts']
  state['fit_attempts'].append(family);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
  model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));model.fit(inputs[family][0],ytr)
  pred=model.predict(inputs[family][1]);assert np.isfinite(pred).all()
  models[family]=model;predictions[family]=pred;recorder.save_objects(artifact_path='models',**{family:model});recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'signal_date':held.index.get_level_values('datetime'),'prediction':pred,'target':yhe})})
  state['fits_completed'].append(family);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');recorder.log_metrics(completed_fits=len(state['fits_completed']))
 dates=pd.DatetimeIndex(held.index.get_level_values('datetime'));scored=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
 scored['paired_mse_improvement']=(scored.target-scored.baseline_prediction)**2-(scored.target-scored.candidate_prediction)**2
 full=pd.DataFrame({'signal_date':pd.DatetimeIndex(he.index.get_level_values('datetime'))}).merge(scored[['signal_date','paired_mse_improvement']],on='signal_date',how='left',validate='one_to_one')
 assert len(full)==944 and full.paired_mse_improvement.notna().sum()==940 and full.paired_mse_improvement.iloc[-4:].isna().all()
 halves=[scored.loc[scored.signal_date.lt(pd.Timestamp('2025-01-01'))],scored.loc[scored.signal_date.ge(pd.Timestamp('2025-01-01'))]]
 annual=[{'year':int(y),'n':int(len(g)),'paired_mean_mse_improvement':float(g.paired_mse_improvement.mean())} for y,g in scored.groupby(scored.signal_date.dt.year,sort=True)]
 complete=[x for x in annual if x['year'] in cohort['complete_held_years']];assert len(complete)==2 and all(x['n']>=240 for x in complete)
 boot=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempts']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 interval,indices,replicates=full_calendar_bootstrap(full.paired_mse_improvement.to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws'],seed=boot['seed'],alpha=boot['one_sided_alpha']);assert interval['calendar_rows']==944 and interval['scored_rows']==940 and interval['valid_replicates']>=9990
 coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1]);metrics={'baseline_held_mse':float(np.mean((scored.target-scored.baseline_prediction)**2)),'candidate_held_mse':float(np.mean((scored.target-scored.candidate_prediction)**2)),'paired_mean_mse_improvement':float(scored.paired_mse_improvement.mean()),'held_half_means':[float(x.paired_mse_improvement.mean()) for x in halves],'annual':annual,'added_standardized_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
 gates={'coefficient_positive':coefficient>0,'full_held_mean_positive':metrics['paired_mean_mse_improvement']>0,'both_held_halves_mean_positive':all(x>0 for x in metrics['held_half_means']),'each_complete_2024_2025_year_mean_positive':all(x['paired_mean_mse_improvement']>0 for x in complete),'bootstrap_lower_positive':metrics['bootstrap_lower']>0};passed=bool(all(gates.values()))
 scored['quarter']=scored.signal_date.dt.to_period('Q').astype(str);quarters=[{'quarter':q,'sessions':len(g),'paired_mean_mse_improvement':float(g.paired_mse_improvement.mean())} for q,g in scored.groupby('quarter',sort=True)]
 recorder.save_objects(artifact_path='analysis',scored=scored,full_eval_calendar=full,metrics=metrics,gates=gates,quarters=quarters,bootstrap_replicates=replicates,bootstrap_indices=indices)
 recorder.log_metrics(paired_mse_improvement=metrics['paired_mean_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'decision':'conditional_development_TSMC_guidance_H5_information_pass_first_public_pending_no_account' if passed else 'not_promoted_fixed_TSMC_guidance_growth_H5_information_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'preflight':report,'information_gate_pass':passed,'metrics':metrics,'gates':gates,'quarterly_information_attribution':quarters,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'held_market_history_reused':True,'historical_first_public_exact_clock_qualified':False,'independent_forward_qualified_days':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'artifact_policy':plan['artifact_policy'],'limitations':['SEC original 6-K sources and next-session clocks are conservative but exact issuer first-public time and full economic semantic revision history remain unqualified.','Reused 2023-2026 held market history is development feedback, not independent validation.','Information MSE is not a full funded 0bp Tiered Stage1 account or SOXX net return.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='tsmc-guidance-info-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:transfer=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert transfer.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  members=[m for m in source.getmembers() if m.isfile()];names=[m.name for m in members]
  assert any(x.endswith('/artifacts/models/baseline') for x in names) and any(x.endswith('/artifacts/models/candidate') for x in names)
  for member in members:assert (card/member.name).is_file() and source.extractfile(member).read()==(card/member.name).read_bytes()
summary['nas_evidence']=evidence;summary['nas_scoped_restore_verified']=True;(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(summary['decision'],metrics,gates,state['recorder_id'])

not_promoted_fixed_TSMC_guidance_growth_H5_information_failed {'baseline_held_mse': 0.0025262669365982205, 'candidate_held_mse': 0.002529224025152378, 'paired_mean_mse_improvement': -2.957088554156739e-06, 'held_half_means': [3.6705882799204444e-06, -1.0553193053487212e-05], 'annual': [{'year': 2023, 'n': 250, 'paired_mean_mse_improvement': 1.2083591001528816e-05}, {'year': 2024, 'n': 252, 'paired_mean_mse_improvement': -4.675644578818019e-06}, {'year': 2025, 'n': 250, 'paired_mean_mse_improvement': -1.3231368378203837e-05}, {'year': 2026, 'n': 188, 'paired_mean_mse_improvement': -6.991789696151276e-06}], 'added_standardized_coefficient': -0.0010703020047631235, 'bootstrap_lower': -2.564096027708037e-05, 'bootstrap_upper': 1.9597576940325693e-05, 'bootstrap': {'paired_mean_loss_improvement': -2.9570885541567386e-06, 'lower_bound': -2.564096027708037e-05, 'upper_bound': 1.9597576940325693e-05, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 944, 'scored_rows': 940}}

## 中文结论
# 原 Stage1 39 项 + TSMC 原始营收指引增长：H5 信息检验

冻结原 Stage1 39 项输入为对照，只增加原始 SEC 6-K 的下一季营收指引中点相对当季已报营收增长一个坐标。训练成熟日 3014、复用历史评价成熟日 940；评价完整 944 日保留末端 4 个未知标签。训练中有 18 个来源未知日，由训练期中位数插补；评价来源 940/940 已知。新坐标与旧 39 项训练期最大逐项相关绝对值 0.1168，未因重复输入而预检停下。

固定两次 Ridge 与一次完整日历区块检验后，对照评价 MSE 0.002526267，候选 0.002529224，成对改善 -2.95708855e-06。新增标准化系数 -0.001070302 与预期正号相反；两个评价半段分别 +3.67058828e-06、-1.05531931e-05，2024/2025 完整年均为负，校正区块下界 -2.56409603e-05。**全部预定信息门失败**，决定 `not_promoted_fixed_TSMC_guidance_growth_H5_information_failed`，不改符号、季度季节控制、持有期或币种公式救援，0 新资金账户。

Qlib Recorder `6d80c635f005430497d086d99a5298fa` 的指标、门槛及 944/940 日历读回一致；Yanxuan NAS 归档 SHA-256 `c7e5f2b8d90dc61a01c7bf55d2bcdc826b69113e1403a00b3db7ff0e94e77f8d` 经远端哈希与限定逐文件恢复。SEC 原始 6-K 的下一交易日使用是保守时钟，仍不证明发行人首次公开时间；2023–2026 历史被反复使用，只能作开发反馈。没有 Stage1 衍生完整 0bp Tiered 账户、2/5bp 或独立前向资格，SOXX v2 目标未达成。
